# 🏆 Simulation Coupe du Monde 2026
## Système de prédiction basé sur des données réelles

> **Objectif :** Simuler l'intégralité de la CdM 2026 — phase de groupes, Round of 32, quarts, demis, finale — à partir d'un modèle de score composite et d'une simulation Monte Carlo (10 000 itérations).
|

### Format CdM 2026
- 48 équipes · 12 groupes de 4 · Top 2 de chaque groupe + 8 meilleurs 3ᵉ = **32 équipes**
- Round of 32 → 1/4 de finale → 1/2 finale → Finale (MetLife Stadium, 19 juillet 2026)

---

## 1. Importation des bibliothèques

In [11]:
import pandas as pd
import numpy as np

## 2. Chargement des données

Lecture des fichiers sources nécessaires à la simulation :

- `valeurs_marchandes.csv` : valeurs marchandes des équipes
- `resultats_depuis_2021.csv` : résultats des matches depuis 2021
- `meilleure_defense.csv` : indicateurs défensifs
- `classement_fifa.csv` : classement FIFA utilisé comme feature
- `buts_marques_depuis_2021.csv` : buts marqués depuis 2021

Chargement dans des DataFrame pandas pour préparation ultérieure.

In [20]:
df_valeurs = pd.read_csv('valeurs_marchandes.csv')
df_resultats = pd.read_csv('resultats_depuis_2021.csv')
df_defenses = pd.read_csv('meilleure_defense.csv')
df_classement= pd.read_csv('classement_fifa.csv')
df_buts = pd.read_csv('buts_marques_depuis_2021.csv')

## 3. Fusion et préparation des données

On construit un `DataFrame` maître (`df_master`) en fusionnant les sources sur la colonne `team`.

- Fusion `classement` × `valeurs_marchandes` → `df_intermediaire`
- Ajout des `resultats` → `df_int`
- Ajout des statistiques défensives → `df_int_2`
- Ajout des buts marqués → `df_master`

Affichage des premières lignes pour vérification.

In [ ]:
df_intermediaire=df_classement.merge(df_valeurs,on='team')
df_int=df_intermediaire.merge(df_resultats,on='team')
df_int_2=df_int.merge(df_defenses,on='team')
df_master=df_int_2.merge(df_buts,on='team')
df_master.head()

,team,classement_fifa,valeur_marchande_euros,matchs_officiels_joues,victoires,nuls,defaites,buts_encaisses_depuis_2021,buts_marques_depuis_2021
0,France,1,1440000000,55,38,9,8,44,102
1,Espagne,2,1390000000,54,40,8,6,47,107
2,Argentine,3,755000000,57,43,8,6,42,112
3,Angleterre,4,1720000000,57,37,11,9,55,108
4,Portugal,5,920000000,50,33,9,8,53,84


## 4. Calcul des indicateurs de performance

Création de nouvelles colonnes pour normaliser les performances de chaque équipe :

- `taux_victoires` = victoires / matchs_officiels_joues
- `taux_invincibilite` = (victoires + nuls) / matchs_officiels_joues
- `buts_encaisses_par_match` = buts_encaisses_depuis_2021 / matchs_officiels_joues
- `buts_marques_par_match` = buts_marques_depuis_2021 / matchs_officiels_joues

Affichage des premières lignes pour vérification.

In [31]:
df_master["taux_victoires"]=df_master["victoires"]/df_master["matchs_officiels_joues"]
df_master["taux_invincibilite"]=(df_master["victoires"]+df_master['nuls'])/df_master["matchs_officiels_joues"]
df_master["buts_encaisses_par_match"]=df_master["buts_encaisses_depuis_2021"]/df_master["matchs_officiels_joues"]
df_master["buts_marques_par_match"]=df_master["buts_marques_depuis_2021"]/df_master["matchs_officiels_joues"]
df_master.head()

,team,classement_fifa,valeur_marchande_euros,matchs_officiels_joues,victoires,nuls,defaites,buts_encaisses_depuis_2021,buts_marques_depuis_2021,taux_victoires,taux_invincibilite,buts_encaisses_par_match,buts_marques_par_match
0,France,1,1440000000,55,38,9,8,44,102,0.690909,0.854545,0.800000,1.854545
1,Espagne,2,1390000000,54,40,8,6,47,107,0.740741,0.888889,0.870370,1.981481
2,Argentine,3,755000000,57,43,8,6,42,112,0.754386,0.894737,0.736842,1.964912
3,Angleterre,4,1720000000,57,37,11,9,55,108,0.649123,0.842105,0.964912,1.894737
4,Portugal,5,920000000,50,33,9,8,53,84,0.660000,0.840000,1.060000,1.680000
